In [25]:
# Bloco 1 - Importar a base de escolas com cidade e estado

from pathlib import Path
import pandas as pd


def obter_pasta_do_notebook():
    return Path.cwd()


def importar_csv_mesma_pasta(nome_arquivo, nome_base, colunas=None, sep="," , engine=None):
    try:
        pasta_notebook = obter_pasta_do_notebook()
        caminho_arquivo = pasta_notebook / nome_arquivo

        argumentos = {
            "filepath_or_buffer": caminho_arquivo,
            "sep": sep,
            "encoding": "utf-8-sig",
            "usecols": colunas
        }

        if engine is not None:
            argumentos["engine"] = engine
        else:
            argumentos["low_memory"] = False

        df = pd.read_csv(**argumentos)
        df.columns = df.columns.str.strip()

        print(f"Base '{nome_base}' importada com sucesso!")
        print(f"Arquivo importado: {caminho_arquivo}")
        print(f"Quantidade de linhas: {df.shape[0]}")
        print(f"Quantidade de colunas: {df.shape[1]}")
        print()

        return df

    except FileNotFoundError:
        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")
        return None

    except Exception as erro:
        print(f"Erro ao importar a base '{nome_base}'.")
        print(f"Detalhes do erro: {erro}")
        return None


def inserir_cidade_estado(df_escolas, df_municipios):

    df_escolas = df_escolas.copy()
    df_municipios = df_municipios.copy()

    df_escolas.columns = df_escolas.columns.str.strip()
    df_municipios.columns = df_municipios.columns.str.strip()

    colunas_municipios_necessarias = [
        "Código Município Completo",
        "Nome_Município",
        "Nome_UF"
    ]

    colunas_faltantes = [
        coluna
        for coluna in colunas_municipios_necessarias
        if coluna not in df_municipios.columns
    ]

    if colunas_faltantes:
        print("Erro nas colunas da base de municípios.")
        print(colunas_faltantes)
        return None

    df_escolas["codigo_cidade"] = pd.to_numeric(
        df_escolas["codigo_cidade"],
        errors="coerce"
    ).astype("Int64")

    df_municipios["Código Município Completo"] = pd.to_numeric(
        df_municipios["Código Município Completo"],
        errors="coerce"
    ).astype("Int64")

    municipios = (
        df_municipios[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ]
        .drop_duplicates(subset=["Código Município Completo"])
        .rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )
    )

    df_escolas = df_escolas.merge(
        municipios,
        on="codigo_cidade",
        how="left"
    )

    colunas = list(df_escolas.columns)

    if "nome_cidade" in colunas:
        colunas.remove("nome_cidade")
    if "estado" in colunas:
        colunas.remove("estado")

    if "codigo_cidade" in colunas:
        posicao = colunas.index("codigo_cidade")
        colunas.insert(posicao + 1, "nome_cidade")
        colunas.insert(posicao + 2, "estado")

    df_escolas = df_escolas[colunas]

    print("Cidade e estado inseridos com sucesso!")
    print(
        "Registros sem município:",
        df_escolas["nome_cidade"].isna().sum()
    )

    return df_escolas


nome_arquivo_escolas = "script_02_escolas.csv"
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

colunas_escolas = [
    "codigo_cidade",
    "id_escola",
    "nome",
    "mec_co_entidade",
    "mec_co_uf",
    "mec_co_municipio",
    "mec_no_entidade",
    "mec_tp_dependencia",
    "mec_tp_localizacao",
    "mec_in_regular",
    "mec_in_eja",
    "mec_in_profissionalizante",
    "mec_in_especial_exclusiva",
    "loc_latitude",
    "loc_longitude",
    "loc_cep",
    "loc_endereco",
    "contato_responsavel",
    "contato_telefone",
    "contato_email",
    "horario_matutino",
    "horario_vespertino",
    "horario_noturno",
    "ensino_superior",
    "ensino_medio",
    "ensino_fundamental",
    "ensino_pre_escola",
    "mec_tp_localizacao_diferenciada",
    "codigo_escola_firebase",
    "alterado_por",
    "dt_criacao",
    "criado_por",
    "dt_alteracao",
    "horario_integral",
    "informacoes_adicionais",
    "nome_unidade_anexa",
    "tipo_unidade"
]

df_escolas = importar_csv_mesma_pasta(
    nome_arquivo_escolas,
    "escolas",
    colunas=colunas_escolas,
    sep="," 
)

df_municipios = importar_csv_mesma_pasta(
    nome_arquivo_municipios,
    "municípios",
    sep=None,
    engine="python"
)

if df_escolas is not None and df_municipios is not None:

    df_escolas_transformado = inserir_cidade_estado(
        df_escolas,
        df_municipios
    )

    if df_escolas_transformado is not None:
        display(df_escolas_transformado)

        pasta_notebook = obter_pasta_do_notebook()
        nome_arquivo_saida = "script_02_escolas_consolidado.csv"
        caminho_arquivo_saida = pasta_notebook / nome_arquivo_saida

        df_escolas_transformado.to_csv(
            caminho_arquivo_saida,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\nArquivo exportado com sucesso!")
        print(f"Caminho: {caminho_arquivo_saida}")


Base 'escolas' importada com sucesso!
Arquivo importado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_02_escolas.csv
Quantidade de linhas: 65860
Quantidade de colunas: 37

Base 'municípios' importada com sucesso!
Arquivo importado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Cidade e estado inseridos com sucesso!
Registros sem município: 291


,codigo_cidade,nome_cidade,estado,id_escola,nome,mec_co_entidade,mec_co_uf,mec_co_municipio,mec_no_entidade,mec_tp_dependencia,...,mec_tp_localizacao_diferenciada,codigo_escola_firebase,alterado_por,dt_criacao,criado_por,dt_alteracao,horario_integral,informacoes_adicionais,nome_unidade_anexa,tipo_unidade
0,1100122,Ji-Paraná,Rondônia,0,Jorge Teixeira de Oliveira,NaN,11,1100122,Jorge Teixeira de Oliveira,2,...,7,oLeADwoQ6dtbDwcfHgfG,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
1,1100148,Nova Brasilândia D'Oeste,Rondônia,0,AURÉLIO BUARQUE DE HOLANDA FERREIRA,NaN,11,1100148,AURÉLIO BUARQUE DE HOLANDA FERREIRA,2,...,7,3NoiHngRMOljg7g09sz9,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
2,1100205,Porto Velho,Rondônia,0,EEEFM. FERNANDO DE SOUZA GOMES,NaN,11,1100205,EEEFM. FERNANDO DE SOUZA GOMES,2,...,7,2W6TCBNpG3txFvKkafuN,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
3,3163409,São José do Goiabal,Minas Gerais,52328122,CENTRO UNIVERSITARIO CATOLICA DO LESTE DE MINA...,331801.0,31,3131307,CENTRO UNIVERSITARIO CATOLICA DO LESTE DE MINA...,4,...,7,NaN,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
4,3304151,Quissamã,Rio de Janeiro,33016550,CE VISCONDE DE QUISSAMA,33016550.0,33,3304151,CE VISCONDE DE QUISSAMA,2,...,7,33016550,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65855,2308104,Mauriti,Ceará,52336086,EEFTI. PROFESSORA MARIA STELLA MARANHÃO,23160799.0,23,2308104,EEFTI. PROFESSORA MARIA STELLA MARANHÃO,3,...,7,NaN,NaN,NaN,NaN,NaN,S,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
65856,2108405,Peri Mirim,Maranhão,52336101,UMI CARNEIRO DE FREITAS,21044767.0,21,2108405,UMI CARNEIRO DE FREITAS,3,...,7,NaN,NaN,NaN,NaN,NaN,S,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
65857,2108405,Peri Mirim,Maranhão,52336102,CRECHE VICENCIA GUIMÃRES,21269211.0,21,2108405,CRECHE VICENCIA GUIMÃRES,3,...,7,NaN,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
65858,2108405,Peri Mirim,Maranhão,52336103,EM TIAGO PEREIRA,21292760.0,21,2108405,EM TIAGO PEREIRA,3,...,7,NaN,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ



Arquivo exportado com sucesso!
Caminho: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_02_escolas_consolidado.csv


In [26]:
# Bloco 2 - Preparar a base de escolas para consolidação

df_consolidacao = df_escolas_transformado.copy()

df_consolidacao["estado"] = (
    df_consolidacao["estado"]
    .fillna("Não identificado")
)

colunas_numericas = [
    "codigo_cidade",
    "id_escola",
    "mec_co_entidade",
    "mec_co_uf",
    "mec_co_municipio",
    "mec_tp_dependencia",
    "mec_tp_localizacao",
    "mec_tp_localizacao_diferenciada"
]

for coluna in colunas_numericas:
    df_consolidacao[coluna] = pd.to_numeric(
        df_consolidacao[coluna],
        errors="coerce"
    ).astype("Int64")

colunas_booleanas = [
    "mec_in_regular",
    "mec_in_eja",
    "mec_in_profissionalizante",
    "mec_in_especial_exclusiva",
    "horario_matutino",
    "horario_vespertino",
    "horario_noturno",
    "horario_integral",
    "ensino_superior",
    "ensino_medio",
    "ensino_fundamental",
    "ensino_pre_escola"
]

for coluna in colunas_booleanas:
    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .fillna("N")
        .astype(str)
        .str.strip()
        .str.upper()
    )

print("Base de escolas preparada para consolidação.")

display(df_consolidacao.head())

Base de escolas preparada para consolidação.


,codigo_cidade,nome_cidade,estado,id_escola,nome,mec_co_entidade,mec_co_uf,mec_co_municipio,mec_no_entidade,mec_tp_dependencia,...,mec_tp_localizacao_diferenciada,codigo_escola_firebase,alterado_por,dt_criacao,criado_por,dt_alteracao,horario_integral,informacoes_adicionais,nome_unidade_anexa,tipo_unidade
0,1100122,Ji-Paraná,Rondônia,0,Jorge Teixeira de Oliveira,<NA>,11,1100122,Jorge Teixeira de Oliveira,2,...,7,oLeADwoQ6dtbDwcfHgfG,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
1,1100148,Nova Brasilândia D'Oeste,Rondônia,0,AURÉLIO BUARQUE DE HOLANDA FERREIRA,<NA>,11,1100148,AURÉLIO BUARQUE DE HOLANDA FERREIRA,2,...,7,3NoiHngRMOljg7g09sz9,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
2,1100205,Porto Velho,Rondônia,0,EEEFM. FERNANDO DE SOUZA GOMES,<NA>,11,1100205,EEEFM. FERNANDO DE SOUZA GOMES,2,...,7,2W6TCBNpG3txFvKkafuN,NaN,NaN,NaN,NaN,N,NaN,NaN,MATRIZ
3,3163409,São José do Goiabal,Minas Gerais,52328122,CENTRO UNIVERSITARIO CATOLICA DO LESTE DE MINA...,331801,31,3131307,CENTRO UNIVERSITARIO CATOLICA DO LESTE DE MINA...,4,...,7,NaN,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ
4,3304151,Quissamã,Rio de Janeiro,33016550,CE VISCONDE DE QUISSAMA,33016550,33,3304151,CE VISCONDE DE QUISSAMA,2,...,7,33016550,NaN,NaN,NaN,NaN,N,"{""mec_in"":{""quilombola"":""N"",""indigena"":""N"",""ed...",NaN,MATRIZ


In [27]:
# Bloco 3 - Função para consolidar modalidades por estado

def consolidar_modalidades_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_escolas=("codigo_cidade", "size"),
            regular=("mec_in_regular", lambda x: (x == "S").sum()),
            eja=("mec_in_eja", lambda x: (x == "S").sum()),
            profissionalizante=("mec_in_profissionalizante", lambda x: (x == "S").sum()),
            especial_exclusiva=("mec_in_especial_exclusiva", lambda x: (x == "S").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de modalidades por estado:")
    display(tabela)

    return tabela

In [28]:
# Bloco 4 - Consolidar modalidades

tabela_modalidades = consolidar_modalidades_por_estado(
    df_consolidacao
)

Tabela consolidada de modalidades por estado:


,estado,total_escolas,regular,eja,profissionalizante,especial_exclusiva
0,Acre,593,548,88,2,36
1,Alagoas,1293,1227,495,66,229
2,Amapá,82,79,18,0,14
3,Amazonas,1714,1624,391,64,17
4,Bahia,7178,6780,1936,155,687
5,Ceará,3705,3496,665,121,335
6,Distrito Federal,89,82,3,0,7
7,Espírito Santo,862,839,60,21,59
8,Goiás,1756,1640,199,57,86
9,Maranhão,3788,3676,971,34,215


In [29]:
# Bloco 5 - Função para consolidar ensino por estado

def consolidar_ensino_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_escolas=("codigo_cidade", "size"),
            ensino_superior=("ensino_superior", lambda x: (x == "S").sum()),
            ensino_medio=("ensino_medio", lambda x: (x == "S").sum()),
            ensino_fundamental=("ensino_fundamental", lambda x: (x == "S").sum()),
            ensino_pre_escola=("ensino_pre_escola", lambda x: (x == "S").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de ensino por estado:")
    display(tabela)

    return tabela


tabela_ensino = consolidar_ensino_por_estado(
    df_consolidacao
)

Tabela consolidada de ensino por estado:


,estado,total_escolas,ensino_superior,ensino_medio,ensino_fundamental,ensino_pre_escola
0,Acre,593,0,92,430,204
1,Alagoas,1293,6,186,885,737
2,Amapá,82,0,0,65,52
3,Amazonas,1714,0,748,1351,606
4,Bahia,7178,67,883,5202,4155
5,Ceará,3705,74,450,2591,2153
6,Distrito Federal,89,0,10,77,33
7,Espírito Santo,862,2,117,588,452
8,Goiás,1756,57,437,1161,936
9,Maranhão,3788,17,461,2987,2364


In [30]:
# Bloco 7 - Função para consolidar horários por estado

def consolidar_horarios_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_escolas=("codigo_cidade", "size"),
            horario_matutino=("horario_matutino", lambda x: (x == "S").sum()),
            horario_vespertino=("horario_vespertino", lambda x: (x == "S").sum()),
            horario_noturno=("horario_noturno", lambda x: (x == "S").sum()),
            horario_integral=("horario_integral", lambda x: (x == "S").sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de horários por estado:")
    display(tabela)

    return tabela

In [31]:
# Bloco 8 - Consolidar horários

tabela_horarios = consolidar_horarios_por_estado(
    df_consolidacao
)

Tabela consolidada de horários por estado:


,estado,total_escolas,horario_matutino,horario_vespertino,horario_noturno,horario_integral
0,Acre,593,238,135,18,9
1,Alagoas,1293,970,901,490,314
2,Amapá,82,82,67,12,2
3,Amazonas,1714,1367,1204,605,16
4,Bahia,7178,5528,4991,1876,1217
5,Ceará,3705,2941,2787,665,1155
6,Distrito Federal,89,76,77,5,12
7,Espírito Santo,862,728,628,57,121
8,Goiás,1756,1337,1235,285,161
9,Maranhão,3788,3092,2747,901,189


In [32]:
# Bloco 11 - Consolidar todas as tabelas e exportar para Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):

    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):

        tabela_temp = tabela.copy()

        if indice > 0 and "total_escolas" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_escolas"])

        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    tabela_final = (
        tabela_final
        .sort_values("estado")
        .reset_index(drop=True)
    )

    linha_brasil = {"estado": "Brasil"}

    for coluna in tabela_final.columns:
        if coluna != "estado":
            linha_brasil[coluna] = tabela_final[coluna].sum()

    tabela_final = pd.concat(
        [
            tabela_final,
            pd.DataFrame([linha_brasil])
        ],
        ignore_index=True
    )

    return tabela_final


tabelas_para_consolidar = [
    tabela_modalidades,
    tabela_ensino,
    tabela_horarios
]

tabela_consolidada_final = consolidar_tabelas_por_estado(
    tabelas_para_consolidar
)

pasta_notebook = obter_pasta_do_notebook()

nome_arquivo_excel = "Consolidação 02 - script_02_escolas.xlsx"

caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")
print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)

Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: c:\Users\Matheus Henrique\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 02 - script_02_escolas.xlsx


,estado,total_escolas,regular,eja,profissionalizante,especial_exclusiva,ensino_superior,ensino_medio,ensino_fundamental,ensino_pre_escola,horario_matutino,horario_vespertino,horario_noturno,horario_integral
0,Acre,593,548,88,2,36,0,92,430,204,238,135,18,9
1,Alagoas,1293,1227,495,66,229,6,186,885,737,970,901,490,314
2,Amapá,82,79,18,0,14,0,0,65,52,82,67,12,2
3,Amazonas,1714,1624,391,64,17,0,748,1351,606,1367,1204,605,16
4,Bahia,7178,6780,1936,155,687,67,883,5202,4155,5528,4991,1876,1217
5,Ceará,3705,3496,665,121,335,74,450,2591,2153,2941,2787,665,1155
6,Distrito Federal,89,82,3,0,7,0,10,77,33,76,77,5,12
7,Espírito Santo,862,839,60,21,59,2,117,588,452,728,628,57,121
8,Goiás,1756,1640,199,57,86,57,437,1161,936,1337,1235,285,161
9,Maranhão,3788,3676,971,34,215,17,461,2987,2364,3092,2747,901,189
